In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/products.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/users.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/order_items.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/reviews.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/orders.csv
/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/events.csv


In [2]:
import sqlite3

# ECサイトにおける消費者行動分析と商品推薦の提案

## 分析目的

本分析では、ECサイトにおけるユーザー行動データを用いて、消費者がどのような商品に興味を持ち、どのような条件で購買に至るのかを分析する。

最終的には、分析結果をもとに「どのような商品が売れやすいのか」を整理し、ユーザーに対する商品推薦や、ECサイト運営における改善提案につなげることを目的とする。

---

# 現時点での分析方針

## 1. ユーザーの興味・購買傾向を把握する

現時点では、ユーザーの興味を直接表す情報が限定的であるため、以下の特徴量を利用して傾向を分析する。

### 使用を想定している特徴量

| 特徴量 | 想定する意味 |
|---|---|
| 購買数 | 実際の需要・人気 |
| レビュー評価 | 商品満足度 |
| 価格 | 購買ハードル |
| ウィッシュリスト追加数 | 興味・比較検討段階 |
| 閲覧数(view) | 商品への関心 |
| カート追加(cart) | 購買意欲 |

---

# 仮説

現段階では、

> 「購買数とレビュー評価がともに高い商品は、人気商品である」

という仮説を置いて分析を進める。

まずは、この仮説をもとに上位商品を確認し、どのようなカテゴリ・価格帯・特徴を持つ商品が人気なのかを観察する。

---

# 分析ロードマップ

## Phase1 : 人気商品の把握

まずは以下を確認する。

- 購買数上位の商品
- レビュー評価の高い商品
- 購買数と評価の両方が高い商品
- 上位10商品の特徴

ここでは、人気商品の傾向を把握することを目的とする。

---

## Phase2 : 興味・購買行動の分析

ユーザー行動を以下のような段階として整理する。

| 行動 | 意味 |
|---|---|
| view | 興味 |
| wishlist | 比較・検討 |
| cart | 購買意欲 |
| purchase | 実購入 |
| review | 満足度 |

これらをもとに、

- 興味は高いが購入されない商品
- 購入率の高い商品
- 特定カテゴリの行動傾向

などを分析する。

---

## Phase3 : 売れる商品の条件分析

人気商品や購買率の高い商品の特徴を整理し、

- 価格帯
- レビュー傾向
- カテゴリ
- ユーザー行動

などから、「売れやすい商品の条件」を分析する。

---

## Phase4 : 商品推薦・改善提案

分析結果をもとに、

- ユーザーごとの商品推薦
- 類似商品の提案
- 人気商品の特徴整理
- ECサイト改善案

などにつなげることを目指す。

---

# 現時点で重視していること

本分析では、単純な売上集計だけではなく、

> 「ユーザーがどのような行動を経て購入に至るのか」

という消費者行動の流れを重視して分析を進める。

また、分析前の段階として、

- どの特徴量を使用するか
- 各行動をどのように定義するか
- どの仮説を置くか

といった分析設計も重視する。

# Day1


## 目的

ECサイトのユーザー行動データを用いて、
商品がどのような条件で購買されるのかを分析する。

特に以下の流れを明らかにすることを目的とする：

- 興味（view / wishlist）
- 検討（cart）
- 購買（purchase）

この一連の行動構造を商品単位で可視化する。

## アプローチ

データは複数テーブルに分かれているため、
SQL（DuckDB）を用いて商品単位の分析テーブルを構築する。

Pythonでは以下の役割で処理を分担：

- SQL：データ統合・集計（前処理）
- pandas：確認・分析
- 可視化：傾向の把握

## 前処理（SQL）

products、order_items、reviews、eventsを統合し、
商品単位で以下の指標を作成した：

- purchase_count（購買数）
- avg_rating（レビュー評価）
- view_count（閲覧数）
- wishlist_count（興味）
- cart_count（購買意欲）

これにより、商品ごとの行動構造を1つのテーブルに統合した。

■ EC商品分析用・前処理SQL

In [3]:
import pandas as pd
import duckdb

In [4]:
#Fileの読み込み
products = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/products.csv")
users = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/users.csv")
order_items = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/order_items.csv")
reviews = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/reviews.csv")
orders = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/orders.csv")
events = pd.read_csv("/kaggle/input/datasets/abhayayare/e-commerce-dataset/ecommerce_dataset/events.csv")

In [5]:
#DuckDBへの接続を作成
con = duckdb.connect()

In [6]:
#SQLで分析するデータファイルの登録
con.register("products", products)
con.register("users", users)
con.register("order_items", order_items)
con.register("reviews", reviews)
con.register("orders", orders)
con.register("events", events)

In [7]:
#冒頭5行のproduct_idとその注文頻度を集計
query = """
SELECT product_id, COUNT(*) AS cnt
FROM order_items
GROUP BY product_id
ORDER BY cnt DESC
"""

con.execute(query).df().head()

,product_id,cnt
0,P000519,38
1,P000762,37
2,P001164,36
3,P001354,35
4,P001239,35


In [8]:
#各商品のカテゴリ・価格と、注文回数をまとめて、多い順に並べ集計
query = """
SELECT
    p.product_id,
    p.category,
    p.price,
    COUNT(oi.order_id) AS cnt
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY p.product_id, p.category, p.price
ORDER BY cnt DESC
"""

con.execute(query).df()

,product_id,category,price,cnt
0,P000519,Electronics,769.71,38
1,P000762,Clothing,185.46,37
2,P001164,Clothing,18.65,36
3,P001354,Toys,96.63,35
4,P000007,Toys,81.17,35
...,...,...,...,...
1995,P001129,Home & Kitchen,305.16,10
1996,P000168,Sports,298.04,10
1997,P001416,Groceries,26.31,10
1998,P001131,Beauty,127.21,9


① カテゴリ別平均購入数

In [9]:
query = """
SELECT
    category,
    AVG(cnt) AS avg_purchase,
    COUNT(*) AS product_count
FROM (
SELECT
    p.product_id,
    p.category,
    p.price,
    COUNT(oi.order_id) AS cnt
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY p.product_id, p.category, p.price
ORDER BY cnt DESC
) t
GROUP BY category
ORDER BY avg_purchase DESC;
"""

con.execute(query).df()

,category,avg_purchase,product_count
0,Pet Supplies,22.306604,212
1,Books,22.216495,194
2,Sports,22.153005,183
3,Toys,22.145540,213
4,Beauty,21.966184,207
5,Automotive,21.943005,193
6,Clothing,21.610329,213
7,Home & Kitchen,21.236181,199
8,Groceries,21.125683,183
9,Electronics,20.876847,203


② 価格帯ごとの分析

In [10]:
query = """
SELECT
    category,
    CASE
        WHEN price < 50 THEN 'low'
        WHEN price < 200 THEN 'mid'
        ELSE 'high'
    END AS price_range,
    AVG(cnt) AS avg_purchase
FROM (
    SELECT
        p.product_id,
        p.category,
        p.price,
        COUNT(oi.order_id) AS cnt
    FROM products p
    JOIN order_items oi
        ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.category, p.price
) t
GROUP BY category, price_range
ORDER BY avg_purchase DESC;
"""

con.execute(query).df()

,category,price_range,avg_purchase
0,Toys,high,29.000000
1,Automotive,low,23.857143
2,Books,mid,23.325000
3,Home & Kitchen,low,22.583333
4,Pet Supplies,low,22.537736
5,Sports,mid,22.437500
6,Toys,low,22.391753
7,Beauty,low,22.360656
8,Beauty,high,22.166667
9,Pet Supplies,mid,22.075472


## 初期結果

分析の結果、以下の傾向が見られた：

- Toysカテゴリは価格に関係なく購買数が高い
- 多くのカテゴリは価格帯による差が小さい
- Electronicsは比較検討型の行動構造の可能性がある

単純な価格ではなく、カテゴリごとに購買構造が異なる可能性がある。

## 今後の分析

- view → purchaseの転換率分析
- カテゴリ別購買構造の比較
- 価格帯と購買の関係分析
- レコメンドモデルの検討

# 📊 Day2：E-Commerce購買意欲分析プロトタイプ

---

## ■ 目的

本日は「購買意欲の構造」を仮説ベースで数値化し、  
ユーザー行動（view → wishlist → purchase）から  
簡易的な購買意欲スコアを構築することを目的とする。

---

## ■ 今日のゴール

- eventsデータからユーザー×商品単位で行動を集計する  
- view / wishlist / purchase の発生回数を算出する  
- 仮説的な購買意欲スコア（intent_score）を作成する  
- 上位商品・ユーザーの傾向を軽く確認する  

---

## ■ 分析の前提仮説

- view → wishlist → purchase の順に購買意欲は高まる  
- 行動の段階が多いほど「関与度」が高いとみなす  
- 購買意欲は直接観測できないため、行動から代理指標として定義する  

---

## ■ 使用データ

- events（主データ）
- products（必要に応じて結合）

---

## ■ 実装方針（SQL）

1. user_id × product_id 単位で集計  
2. event_typeごとのカウントを作成  
3. CASEまたはSUMでスコア化  
4. intent_scoreを算出  

---

## ■ 仮スコア設計

```text
intent_score =
    1 * view
  + 2 * wishlist
  + 3 * purchase

In [11]:
#ユーザー×商品の購買意欲分析SQL
query = """
WITH event_counts AS (
    SELECT
        user_id,
        product_id,

        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_cnt,
        SUM(CASE WHEN event_type = 'wishlist' THEN 1 ELSE 0 END) AS wishlist_cnt,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_cnt

    FROM events
    GROUP BY user_id, product_id
)

SELECT
    user_id,
    product_id,
    view_cnt,
    wishlist_cnt,
    purchase_cnt,
    (view_cnt + 2 * wishlist_cnt + 3 * purchase_cnt) AS intent_score
FROM event_counts
ORDER BY intent_score DESC;
"""

con.execute(query).df()

,user_id,product_id,view_cnt,wishlist_cnt,purchase_cnt,intent_score
0,U007562,P001490,0.0,1.0,1.0,5.0
1,U004871,P000009,0.0,1.0,1.0,5.0
2,U005708,P000597,1.0,0.0,1.0,4.0
3,U009090,P000130,1.0,0.0,1.0,4.0
4,U008807,P001859,1.0,0.0,1.0,4.0
...,...,...,...,...,...,...
79820,U007448,P001746,0.0,0.0,0.0,0.0
79821,U001552,P000394,0.0,0.0,0.0,0.0
79822,U008061,P000246,0.0,0.0,0.0,0.0
79823,U001186,P000003,0.0,0.0,0.0,0.0


In [12]:
#上位商品を見る（商品単位集計）
query = """
WITH event_counts AS (
    SELECT
        product_id,

        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_cnt,
        SUM(CASE WHEN event_type = 'wishlist' THEN 1 ELSE 0 END) AS wishlist_cnt,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_cnt

    FROM events
    GROUP BY product_id
)

SELECT
    product_id,
    view_cnt,
    wishlist_cnt,
    purchase_cnt,

    (view_cnt * 1 + wishlist_cnt * 2 + purchase_cnt * 3) AS intent_score

FROM event_counts
ORDER BY intent_score DESC;
"""

con.execute(query).df()

,product_id,view_cnt,wishlist_cnt,purchase_cnt,intent_score
0,P000296,40.0,9.0,5.0,73.0
1,P000881,40.0,6.0,6.0,70.0
2,P001771,42.0,8.0,4.0,70.0
3,P001340,39.0,8.0,5.0,70.0
4,P000751,38.0,8.0,5.0,69.0
...,...,...,...,...,...
1995,P001486,14.0,1.0,2.0,22.0
1996,P001015,17.0,1.0,1.0,22.0
1997,P000951,19.0,1.0,0.0,21.0
1998,P001207,15.0,3.0,0.0,21.0


In [13]:
#ユーザーの“意欲層”を作る（セグメント化）
query = """
WITH user_scores AS (
    SELECT
        user_id,
        COUNT(CASE WHEN event_type = 'view' THEN 1 END) AS views,
        COUNT(CASE WHEN event_type = 'wishlist' THEN 1 END) AS wishlists,
        COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS purchases
    FROM events
    GROUP BY user_id
),

scored AS (
    SELECT
        user_id,
        views,
        wishlists,
        purchases,
        (views + 2 * wishlists + 3 * purchases) AS intent_score
    FROM user_scores
)

SELECT *,
    CASE
        WHEN intent_score >= 50 THEN 'high_intent'
        WHEN intent_score >= 20 THEN 'mid_intent'
        ELSE 'low_intent'
    END AS intent_segment

FROM scored
ORDER BY intent_score DESC;
"""

con.execute(query).df()

,user_id,views,wishlists,purchases,intent_score,intent_segment
0,U007388,13,2,2,23,mid_intent
1,U006046,16,2,1,23,mid_intent
2,U008717,7,3,3,22,mid_intent
3,U006536,10,3,2,22,mid_intent
4,U001903,4,0,6,22,mid_intent
...,...,...,...,...,...,...
9990,U007378,0,0,0,0,low_intent
9991,U005345,0,0,0,0,low_intent
9992,U007924,0,0,0,0,low_intent
9993,U002731,0,0,0,0,low_intent


In [14]:
#遷移率ベースの購買意欲モデル
query = """
WITH base AS (
    SELECT
        user_id,
        product_id,

        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_cnt,
        SUM(CASE WHEN event_type = 'wishlist' THEN 1 ELSE 0 END) AS wishlist_cnt,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_cnt

    FROM events
    GROUP BY user_id, product_id
),

funnel AS (
    SELECT
        *,
        -- 遷移率（ゼロ割防止つき）
        CASE WHEN view_cnt > 0 THEN wishlist_cnt * 1.0 / view_cnt ELSE 0 END AS view_to_wishlist_rate,
        CASE WHEN wishlist_cnt > 0 THEN purchase_cnt * 1.0 / wishlist_cnt ELSE 0 END AS wishlist_to_purchase_rate,
        CASE WHEN view_cnt > 0 THEN purchase_cnt * 1.0 / view_cnt ELSE 0 END AS view_to_purchase_rate
    FROM base
)

SELECT
    user_id,
    product_id,
    view_cnt,
    wishlist_cnt,
    purchase_cnt,
    view_to_wishlist_rate,
    wishlist_to_purchase_rate,
    view_to_purchase_rate,

    -- 仮の購買意欲スコア（遷移ベース）
    (
        view_to_wishlist_rate * 0.3 +
        wishlist_to_purchase_rate * 0.5 +
        view_to_purchase_rate * 0.2
    ) AS intent_score

FROM funnel
ORDER BY intent_score DESC;
"""

con.execute(query).df()

,user_id,product_id,view_cnt,wishlist_cnt,purchase_cnt,view_to_wishlist_rate,wishlist_to_purchase_rate,view_to_purchase_rate,intent_score
0,U004871,P000009,0.0,1.0,1.0,0.0,1.0,0.0,0.5
1,U007562,P001490,0.0,1.0,1.0,0.0,1.0,0.0,0.5
2,U007829,P001326,1.0,1.0,0.0,1.0,0.0,0.0,0.3
3,U008438,P001614,1.0,1.0,0.0,1.0,0.0,0.0,0.3
4,U000981,P000731,1.0,1.0,0.0,1.0,0.0,0.0,0.3
...,...,...,...,...,...,...,...,...,...
79820,U007104,P000133,0.0,1.0,0.0,0.0,0.0,0.0,0.0
79821,U003086,P000989,1.0,0.0,0.0,0.0,0.0,0.0,0.0
79822,U004250,P000841,1.0,0.0,0.0,0.0,0.0,0.0,0.0
79823,U002343,P000371,1.0,0.0,0.0,0.0,0.0,0.0,0.0


# 🧠 思考ログ

---

## ■ 思考の出発点

本分析では、単純な売上集計ではなく  
「ユーザーの購買意欲はどのように形成されるのか」という問いからスタートした。

特に、直接観測できない“意図”をデータからどう近似するかに焦点を置いた。

---

## ■ 思考プロセスの変化

当初はSQLによる集計を中心とした分析を想定していたが、  
以下のような視点の変化が発生した：

- 単純な売上や購入回数では行動の意味が見えない
- ユーザー行動は段階的なプロセスとして存在している
- 「view → wishlist → purchase」という遷移構造に注目
- 数値そのものではなく“行動の流れ”に価値があると認識

---

## ■ 構造的再解釈

データを以下のように再定義した：

- view：関心の発生
- wishlist：比較・検討段階
- purchase：意思決定

これにより、単なるイベントログではなく  
**「意思決定プロセスの記録」としてデータを再解釈**する方向へ移行した。

---

## ■ 仮説の形成

購買意欲は直接観測できないため、  
以下のような代理指標として定義した：

- 行動の段階数が多いほど意欲が高い
- view → wishlist → purchase の遷移が強いほど関与度が高い

---

## ■ 分析観点の拡張

分析対象を以下の2軸で捉えるようになった：

- 商品単位の傾向（何が売れているか）
- ユーザー行動の構造（なぜそれが売れるのか）

---

## ■ 認識の変化

- SQLは単なる集計ツールではなく「構造を作るツール」である
- 分析とは結果を出す行為ではなく、仮説を設計する行為である
- データから“意図”を直接読むのではなく、“構造から推定する”必要がある

---

## ■ 現在地の認識

本フェーズは完成された分析ではなく、  
**「行動データの構造化プロトタイプ段階」**であると位置づける。

---

## ■ 今後の課題

- 仮説（購買意欲定義）の妥当性検証
- 指標（intent_score）の改善
- ユーザーセグメント化の導入
- 可視化による傾向確認